# Mandelbrot Zoom
A perpetual zoom into the Seahorse Valley of the Mandelbrot set, rendered as an animated GIF with beautiful coloring.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.colors as mcolors
from matplotlib.animation import FuncAnimation, PillowWriter
from IPython.display import Image, display
import warnings
warnings.filterwarnings('ignore')

In [ ]:
def mandelbrot(xmin, xmax, ymin, ymax, width=600, height=600, max_iter=256):
    x = np.linspace(xmin, xmax, width)
    y = np.linspace(ymin, ymax, height)
    C = x[np.newaxis, :] + 1j * y[:, np.newaxis]
    Z = np.zeros_like(C)
    M = np.zeros(C.shape, dtype=float)
    mask = np.ones(C.shape, dtype=bool)
    for i in range(max_iter):
        Z[mask] = Z[mask] ** 2 + C[mask]
        escaped = mask & (np.abs(Z) > 2)
        smooth_val = i + 1 - np.log2(np.log2(np.abs(Z[escaped]) + 1e-10))
        M[escaped] = smooth_val
        mask[escaped] = False
    M[mask] = max_iter
    return M

In [ ]:
center_x, center_y = -0.743643887037158704752191506114774, 0.131825904205311970493132056385139
zoom_start = 1.5
zoom_factor = 0.925
num_frames = 200

frames_data = []
for i in range(num_frames):
    half_range = zoom_start * (zoom_factor ** i)
    max_iter = min(256 + i * 2, 512)
    xmin = center_x - half_range
    xmax = center_x + half_range
    ymin = center_y - half_range
    ymax = center_y + half_range
    M = mandelbrot(xmin, xmax, ymin, ymax, width=480, height=480, max_iter=max_iter)
    frames_data.append(M)
    if (i + 1) % 40 == 0:
        final_zoom = zoom_start / half_range
        print(f"Computed frame {i + 1}/{num_frames} (zoom: {final_zoom:,.0f}x)")

final_half = zoom_start * (zoom_factor ** (num_frames - 1))
print(f"All frames computed! Final zoom: {zoom_start / final_half:,.0f}x")

In [ ]:
from matplotlib.patches import FancyBboxPatch

colors_list = [
    '#0a0020', '#3b0070', '#8000ff', '#ff00ff',
    '#ff66cc', '#00ccff', '#0044aa', '#6600cc',
    '#ff00aa', '#220044', '#0a0020'
]
cmap = mcolors.LinearSegmentedColormap.from_list('psychedelic', colors_list, N=2048)

fig, ax = plt.subplots(figsize=(6, 6), dpi=80)
fig.subplots_adjust(left=0, right=1, top=1, bottom=0)
ax.set_axis_off()

def add_text_with_bg(ax, x, y, text, fontsize=10, color='white', alpha=0.55):
    t = ax.text(x, y, text, transform=ax.transAxes,
                fontsize=fontsize, color=color,
                fontfamily='monospace', fontweight='bold',
                verticalalignment='top',
                bbox=dict(boxstyle='round,pad=0.3', facecolor='black', alpha=alpha, edgecolor='none'))
    return t

def animate(i):
    ax.clear()
    ax.set_axis_off()
    M = frames_data[i]
    max_iter = min(256 + i * 2, 512)
    norm = mcolors.PowerNorm(0.4, vmin=0, vmax=max_iter)
    ax.imshow(M, cmap=cmap, norm=norm, extent=[0, 1, 0, 1], interpolation='bilinear')

    half_range = zoom_start * (zoom_factor ** i)
    zoom_level = zoom_start / half_range
    cx = center_x
    cy = center_y

    add_text_with_bg(ax, 0.03, 0.97,
        r'$z_{n+1} = z_n^2 + c$',
        fontsize=13, color='#ff66ff')

    add_text_with_bg(ax, 0.03, 0.88,
        f'c = {cx:.6f} + {cy:.6f}i',
        fontsize=9, color='#cc99ff')

    add_text_with_bg(ax, 0.03, 0.80,
        f'Escape: |z| > 2   (bailout radius)',
        fontsize=8, color='#aaaaaa')

    add_text_with_bg(ax, 0.03, 0.08,
        f'Zoom: {zoom_level:,.0f}x  |  Max iter: {max_iter}  |  Frame {i+1}/{num_frames}',
        fontsize=8, color='#00eeff')

    add_text_with_bg(ax, 0.03, 0.15,
        f'Window: [{cx - half_range:.8f}, {cx + half_range:.8f}]',
        fontsize=7, color='#ff99dd')

    return []

anim = FuncAnimation(fig, animate, frames=num_frames, interval=80, blit=True)
anim.save('/tmp/mandelbrot_zoom.gif', writer=PillowWriter(fps=12))
plt.close(fig)
print(f'Animation saved! {num_frames} frames')

In [ ]:
display(Image(filename='/tmp/mandelbrot_zoom.gif'))

In [ ]:
import shutil
shutil.copy('/tmp/mandelbrot_zoom.gif', '/home/notebook/mandelbrot_project/mandelbrot_zoom.gif')
print('GIF size:', round(os.path.getsize('/home/notebook/mandelbrot_project/mandelbrot_zoom.gif') / 1024 / 1024, 1), 'MB')
print()
print('TWO WAYS TO DOWNLOAD:')
print('1. Scroll up to the animation cell → right-click the GIF image → "Save image as..."')
print('2. In the left file panel, look for mandelbrot_zoom.gif under mandelbrot_project/')